In [ ]:
from modelscope import AutoModelForCausalLM, AutoTokenizer
import transformers
import torch
import datetime
import time
import json
import re
import numpy as np

In [13]:
device = "cuda"  # the device to load the model onto
model_name_or_path = "" #model path

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
  model_name_or_path,
  torch_dtype=torch.float16,
  device_map="auto"
)

In [23]:
fewshot_prompt = open("").read() #few-shot prompt path

In [18]:
ANS_RE = re.compile(r"#### (\-?[0-9\.\,]+)")
INVALID_ANS = "[invalid]"
def extract_answer_hf(completion):
    match = ANS_RE.search(completion)
    if match:
        match_str = match.group(1).strip()
        match_str = match_str.replace(",", "")
        return eval(match_str)
    else:
        return INVALID_ANS


def extract_answer(completion):
    try:
        last_number = re.findall(r"\d+", completion)[-1]
        return eval(last_number)
    except:
        return INVALID_ANS


def is_correct(completion, answer):
    gold = extract_answer_hf(answer)
    print("Answer: "+ str(gold) + "-------" + "Completion: "+ str(extract_answer(completion)))
    assert gold != INVALID_ANS, "No ground truth answer found in the document."
    return extract_answer(completion) == gold

In [ ]:
test_data_path = "" #test data path
acc_res = []
acc_answer = []
total_start_time = time.time()
with open(test_data_path, 'r', encoding='utf-8') as f1:
    json_content = json.load(f1)
    for i in json_content:
        each_start_time = time.time()
        prompt = fewshot_prompt +i['query']+ "\n"
        model_inputs = tokenizer([prompt], return_tensors="pt").to(device)
        generated_ids = model.generate(
            model_inputs.input_ids,
            max_new_tokens=512,
            repetition_penalty=1.15
        )
        generated_ids = [
            output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
        ]
        response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
        answer = i["response"]
        acc = is_correct(response, answer)   
        i["completion"] = response
        i["acc"] = acc
        each_end_time = time.time()
        elapsed_time = each_end_time - each_start_time
        current_time = datetime.datetime.now()
        text_to_write = "in case:"+ str(cnt) + "!" +" Current time: " + str(current_time) + " , Elapsed Time: " + str(elapsed_time) + " seconds"  
        print(text_to_write)
        acc_res.append(response)

total_end_time = time.time()
total_time = total_end_time - total_start_time
print("Acc: ", np.mean(acc_answer))
print("Total time: "+ str(total_time) + " seconds!")   

file_path = ""  
with open(file_path, "a") as file:  
    file.write(text_to_write)  